# 01 Data audit

**Status: NOT RUN ON COLAB.** Generated by `scripts/make_notebooks.py`. Any output cells in the repository copy come from a local CPU smoke run (`SMOKE=True`, 20 samples), never from a real run.

| | |
|---|---|
| Purpose | Recompute split sizes, label counts, seen/unseen labels, exact overlap and token lengths from the raw files. |
| Inputs | Dataset files |
| Outputs | `data_audit.json` |
| Runtime (T4 GPU, estimate) | 2 min (CPU is enough) |
| Fills in `thesis_numbers.json` | re-checks A1-A8 |

Run the cells top to bottom. Set `SMOKE = True` in the setup cell for a dry run on 20 samples. Every run writes a manifest with input and output hashes; the last cell prints an evidence table read from the saved files.

In [ ]:
NOTEBOOK = "01_data_audit"
NEEDS_GPU = False      # True: the notebook stops at once if Colab gave no GPU

# ---- setup (same in every notebook) ----
import os, sys, json, ast, time, hashlib, random, platform, subprocess, re
from pathlib import Path
import numpy as np, pandas as pd

SMOKE = os.environ.get("TTPMAP_SMOKE", "0") == "1"      # set SMOKE = True by hand for a 20-sample dry run
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount failed:", type(e).__name__, e)
    if not os.path.isdir("/content/drive/MyDrive") and not SMOKE:
        raise RuntimeError("Google Drive is not mounted. The Colab extension for VS Code cannot mount Drive (known limitation of the extension), "
                           "and these notebooks read their inputs from and save their results to Drive. Open this notebook in the browser instead: "
                           "https://colab.research.google.com > File > Upload notebook (or open it from Drive). See notebooks/COLAB_RUNBOOK.md, section 0.")
BASE = Path("/content/drive/MyDrive/KDU/Thesis") if IN_COLAB else Path(os.environ.get("TTPMAP_ROOT", r"E:\Osamas Paper"))
DATA_DIR, CLS_ROOT, MATCH_ROOT, SHAP_DIR = BASE / "Dataset", BASE / "Exp_1_4_run_tram_2_epoc_30", BASE / "run_match_1", BASE / "Exp_3_0_shap"
MATCH2 = BASE / "run_match_2_saved"                      # NEW folder: matcher retrained WITH saved weights (notebook 04)
OUT = (Path(os.environ["TTPMAP_SMOKE_OUT"]) if os.environ.get("TTPMAP_SMOKE_OUT") else BASE / "showcase_runs") / NOTEBOOK
OUT.mkdir(parents=True, exist_ok=True)
SEEDS = (42, 43, 44)
if IN_COLAB and not SMOKE and not (BASE / "Dataset" / "tram_test.tsv").exists():
    _msg = (f"Project files not found under {BASE}. Upload the folder E:\\Osamas Paper\\_showcase\\drive_upload\\KDU into 'My Drive' "
            "(drive.google.com, same Google account as this Colab session), then run this cell again. See notebooks/COLAB_RUNBOOK.md, section 0.")
    if NOTEBOOK == "00_setup_and_verify": print("WARNING:", _msg)
    else: raise RuntimeError(_msg)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def env_info():
    info = {"python": platform.python_version(), "platform": platform.platform(), "in_colab": IN_COLAB, "smoke": SMOKE}
    try:
        import torch
        info.update(torch=torch.__version__, cuda=torch.cuda.is_available(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
    except ImportError:
        pass
    return info

def write_manifest(run_id, system, seed, config, inputs, outputs, metrics, status):
    """One manifest per run: what went in, what came out, with hashes. Read later by the ingest step."""
    m = {"run_id": run_id, "system": system, "seed": seed, "config_hash": hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:16], "config": config,
         "env": {k: str(v) for k, v in env_info().items()}, "input_sha256": {str(p): sha256(p) for p in inputs if Path(p).exists()},
         "output_sha256": {str(p): sha256(p) for p in outputs if Path(p).exists()}, "metrics": {k: float(v) for k, v in metrics.items()},
         "status": "SMOKE" if SMOKE else status, "written": time.strftime("%Y-%m-%dT%H:%M:%S")}
    (OUT / f"manifest_{run_id}.json").write_text(json.dumps(m, indent=1)); return m

print(f"notebook={NOTEBOOK} | SMOKE={SMOKE} | IN_COLAB={IN_COLAB} | BASE={BASE} | OUT={OUT}")
print(env_info())
import importlib.metadata as _md
print("packages:", {p: (_md.version(p) if p in {d.metadata["Name"].lower() for d in _md.distributions() if d.metadata["Name"]} else "not installed")
                    for p in ("torch", "transformers", "scikit-learn", "numpy", "pandas", "shap", "scipy")})
_gpu = env_info().get("cuda", False)
if NEEDS_GPU and not _gpu and not SMOKE:
    raise RuntimeError("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU > Save, then run this cell again. (Set SMOKE = True only for a dry run.)")
if not NEEDS_GPU and not _gpu:
    print("note: no GPU; this notebook runs on CPU in a few minutes")

In [ ]:
# ---- data, label space, gold matrices (same as Exp_1_4 CELL 4 and Exp_2.1 CELL 2) ----
def _labels(x): return sorted({str(t).strip().upper() for t in ast.literal_eval(x)})
def load_split(name):
    p = DATA_DIR / f"tram_{name}.tsv"
    if p.exists():
        df = pd.read_csv(p, sep="\t").rename(columns={"text1": "text"})
    elif name == "train":                                   # local fallback: the tram slice of combined_train.csv is the same data
        df = pd.read_csv(DATA_DIR / "combined_train.csv"); df = df[df["source_dataset"] == "tram"].rename(columns={"text1": "text"})
    else:
        raise FileNotFoundError(p)
    df = df[["text", "labels"]].copy(); df["labels"] = df["labels"].apply(_labels)
    return df.drop_duplicates(subset=["text"]).reset_index(drop=True) if name == "train" else df.reset_index(drop=True)

train, dev, test = (load_split(s) for s in ("train", "dev", "test"))
desc = pd.read_csv(DATA_DIR / "ttp-desc-mappings.csv", sep="\t")
desc["ttp"] = desc["ttp"].str.lower().str.strip()
desc["description"] = desc["description"].fillna("").str.replace(r"\s+", " ", regex=True).str.replace(r"\bcitation:\s*", "", regex=True).str.strip()
desc = desc.drop_duplicates("ttp"); desc = desc[~desc["ttp"].isin({"t1060", "t1107", "t1151", "t1188"})].reset_index(drop=True)
LABELS = [t.upper() for t in desc["ttp"]]; LABEL_TEXTS = desc["description"].tolist(); L = len(LABELS); lab2i = {l: i for i, l in enumerate(LABELS)}
def to_Y(df):
    Y = np.zeros((len(df), L), dtype=bool)
    for i, ys in enumerate(df["labels"]):
        for y in ys: Y[i, lab2i[y]] = True
    return Y
Y_train, Y_dev, Y_test = to_Y(train), to_Y(dev), to_Y(test)
train_count = Y_train.sum(0); SEEN = train_count > 0; seen_idx = np.where(SEEN)[0]
assert (len(train), len(dev), len(test), L, int(SEEN.sum())) == (3468, 604, 725, 733, 180), "unexpected data shape"
TEST_IDS = [f"test-{i:04d}" for i in range(len(test))]; DEV_IDS = [f"dev-{i:04d}" for i in range(len(dev))]
N_SMOKE = 20
print(f"train={len(train)} dev={len(dev)} test={len(test)} labels={L} seen={int(SEEN.sum())}")

In [ ]:
norm = lambda s: re.sub(r"\s+", " ", str(s).lower()).strip()
nset = {k: set(map(norm, d["text"])) for k, d in (("train", train), ("dev", dev), ("test", test))}
audit = {"n_train": len(train), "n_dev": len(dev), "n_test": len(test), "train_labels": int(SEEN.sum()), "test_labels": int((Y_test.sum(0) > 0).sum()),
         "test_labels_seen": int(((Y_test.sum(0) > 0) & SEEN).sum()), "test_labels_unseen": int(((Y_test.sum(0) > 0) & ~SEEN).sum()),
         "exact_overlap_train_test": len(nset["train"] & nset["test"]), "exact_overlap_train_dev": len(nset["train"] & nset["dev"]), "exact_overlap_dev_test": len(nset["dev"] & nset["test"]),
         "multi_label_test": int((Y_test.sum(1) > 1).sum()), "gold_pairs_test": int(Y_test.sum()), "labels_in_space": L}
try:
    from transformers import AutoTokenizer
    for nm, src in (("roberta", CLS_ROOT / "roberta_pw_s42"), ("secbert", CLS_ROOT / "secbert_pw_s42")):
        tk = AutoTokenizer.from_pretrained(str(src) if (src / "tokenizer.json").exists() else {"roberta": "roberta-base", "secbert": "jackaduma/SecBERT"}[nm])
        tx = list(test["text"])[:N_SMOKE] if SMOKE else list(test["text"])
        ln = np.array([len(x) for x in tk(tx, truncation=False)["input_ids"]])
        audit[f"test_over_128_{nm}"] = int((ln > 128).sum()); audit[f"test_mean_tokens_{nm}"] = float(ln.mean())
except Exception as e:
    print("token lengths skipped:", type(e).__name__, e)
(OUT / "data_audit.json").write_text(json.dumps({"metrics": audit}, indent=1))
write_manifest("data_audit", "data", "-", {"smoke": SMOKE}, [DATA_DIR / "tram_test.tsv", DATA_DIR / "tram_dev.tsv"], [OUT / "data_audit.json"], audit, "DONE")

In [ ]:
# ---- evidence table: read back from the files this notebook wrote (no typed numbers) ----
rows = []
for p in sorted(OUT.glob("*.json")):
    d = json.loads(p.read_text())
    flat = d.get("metrics", d) if isinstance(d, dict) else {}
    for k, v in flat.items():
        if isinstance(v, (int, float)) and not isinstance(v, bool): rows.append({"file": p.name, "key": k, "value": round(float(v), 4)})
ev = pd.DataFrame(rows)
print(f"status: {'SMOKE (not a result)' if SMOKE else 'RUN'} | files in {OUT}: {len(list(OUT.glob('*')))}")
print(ev.to_string(index=False) if len(ev) else "no numeric outputs")